# Notebook 4: Linear Regression Predicting Customer Credit Score
## ST7082CEM : Big Data Management and Data Visualisation
### Smaran Luitel | Student ID: 250087

This notebook applies Linear Regression to predict `CreditScore` from customer demographic, financial, and behavioural features.

**Justification:** CreditScore is a meaningful continuous target because it is shaped by real financial behaviours: account balance, product usage, tenure, and activity status. Unlike EstimatedSalary (which is effectively uniformly random in this dataset, yielding R²≈0), CreditScore has genuine predictive structure as it reflects observable customer attributes. Predicting it answers whether the bank’s own data can reconstruct the credit bureau score, a practically relevant question for risk profiling and customer management.

PySpark’s `LinearRegression` implements distributed L-BFGS optimisation. Ridge regularisation (L2) stabilises coefficients on correlated features.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/user/AppData/Local/Programs/Python/Python314/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = "C:/Users/user/AppData/Local/Programs/Python/Python314/python.exe"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ST7082CEM_Regression_250087") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


## 1. Load Data

In [2]:
df = (
    spark.read.csv("../dataset/Customer-Churn-Records.csv", header=True, inferSchema=True)
    .withColumnRenamed("Satisfaction Score", "SatisfactionScore")
    .withColumnRenamed("Card Type", "CardType")
    .withColumnRenamed("Point Earned", "PointEarned")
)
print(f"Loaded {df.count()} rows")
print("CreditScore distribution:")
df.select(
    F.round(F.mean("CreditScore"), 2).alias("mean"),
    F.round(F.stddev("CreditScore"), 2).alias("stddev"),
    F.min("CreditScore").alias("min"),
    F.max("CreditScore").alias("max")
).show()

Loaded 10000 rows
CreditScore distribution:


+------+------+---+---+
|  mean|stddev|min|max|
+------+------+---+---+
|650.53| 96.65|350|850|
+------+------+---+---+



## 2. Feature Engineering

`CreditScore` is the regression target. All remaining numeric and categorical features are used as predictors.

**Excluded:** RowNumber, CustomerId, Surname (identifiers), Complain (confirmed leakage from Notebook 1), CreditScore (target).

**Predictors:**
- **Numeric:** Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary, SatisfactionScore, PointEarned, Exited
- **Categorical (OHE):** Geography, Gender, CardType

In [3]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

cat_cols = ["Geography", "Gender", "CardType"]
num_cols = ["Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard",
            "IsActiveMember", "EstimatedSalary", "SatisfactionScore", "PointEarned", "Exited"]

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe") for c in cat_cols]

assembler = VectorAssembler(
    inputCols=num_cols + [c + "_ohe" for c in cat_cols],
    outputCol="reg_features_raw",
    handleInvalid="keep"
)
scaler = StandardScaler(
    inputCol="reg_features_raw",
    outputCol="reg_features",
    withMean=True,
    withStd=True
)

reg_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
df_reg = reg_pipeline.fit(df).transform(df)

print("Feature pipeline complete.")
print(f"Numeric features: {num_cols}")
print(f"Categorical (OHE): {cat_cols}")
print("Target: CreditScore")

Feature pipeline complete.
Numeric features: ['Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'SatisfactionScore', 'PointEarned', 'Exited']
Categorical (OHE): ['Geography', 'Gender', 'CardType']
Target: CreditScore


## 3. Train/Test Split

In [4]:
train_reg, test_reg = df_reg.randomSplit([0.8, 0.2], seed=42)

print(f"Train size: {train_reg.count()}")
print(f"Test size:  {test_reg.count()}")

print("\nCreditScore distribution (train set):")
train_reg.select(
    F.round(F.mean("CreditScore"), 2).alias("mean"),
    F.round(F.stddev("CreditScore"), 2).alias("stddev"),
    F.min("CreditScore").alias("min"),
    F.max("CreditScore").alias("max")
).show()

Train size: 8079


Test size:  1921

CreditScore distribution (train set):


+------+------+---+---+
|  mean|stddev|min|max|
+------+------+---+---+
|651.05| 96.58|350|850|
+------+------+---+---+



## 4. Linear Regression Model

**Configuration:**
- Regularisation: Ridge/L2 (`elasticNetParam=0.0`) — prevents coefficient instability on correlated features
- `regParam=0.01` — mild penalty, preserves interpretability
- `maxIter=100` — sufficient for L-BFGS convergence on this dataset size

In [5]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

lr = LinearRegression(
    featuresCol="reg_features",
    labelCol="CreditScore",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_reg)
reg_predictions = lr_model.transform(test_reg)

print("Linear Regression model trained.")
print(f"Intercept: {lr_model.intercept:.4f}")

# Build feature name list matching assembled order
feature_names = num_cols + [f"{c}_ohe_{i}" for c in cat_cols for i in range(3)]
feature_names = feature_names[:len(lr_model.coefficients)]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lr_model.coefficients.toArray()
}).sort_values("coefficient", key=abs, ascending=False)

print("\nTop 10 Coefficients (by absolute magnitude):")
print(coef_df.head(10).to_string(index=False))

Linear Regression model trained.
Intercept: 651.0602

Top 10 Coefficients (by absolute magnitude):
          feature  coefficient
   IsActiveMember     2.597637
           Exited    -2.481030
          Balance     2.110047
    NumOfProducts     1.949633
SatisfactionScore    -1.484783
   CardType_ohe_0     0.775945
      PointEarned    -0.737271
   CardType_ohe_1    -0.602097
           Tenure    -0.582060
  EstimatedSalary    -0.499003


## 5. Evaluation

In [6]:
eval_rmse = RegressionEvaluator(labelCol="CreditScore", predictionCol="prediction", metricName="rmse")
eval_r2   = RegressionEvaluator(labelCol="CreditScore", predictionCol="prediction", metricName="r2")
eval_mae  = RegressionEvaluator(labelCol="CreditScore", predictionCol="prediction", metricName="mae")

rmse = eval_rmse.evaluate(reg_predictions)
r2   = eval_r2.evaluate(reg_predictions)
mae  = eval_mae.evaluate(reg_predictions)

print("=== Linear Regression: CreditScore Prediction ===")
print(f"RMSE:          {rmse:>10.2f}")
print(f"MAE:           {mae:>10.2f}")
print(f"R² (test):     {r2:>10.4f}")
print(f"R² (training): {lr_model.summary.r2:>10.4f}")

print("\nSample: Actual vs Predicted CreditScore")
reg_predictions.select(
    F.col("CreditScore").alias("actual"),
    F.round("prediction", 1).alias("predicted"),
    F.round(F.abs(F.col("CreditScore") - F.col("prediction")), 1).alias("abs_error")
).show(10)

=== Linear Regression: CreditScore Prediction ===
RMSE:               97.15
MAE:                78.59
R² (test):        -0.0047
R² (training):     0.0027

Sample: Actual vs Predicted CreditScore


+------+---------+---------+
|actual|predicted|abs_error|
+------+---------+---------+
|   502|    650.7|    148.7|
|   822|    654.6|    167.4|
|   501|    661.7|    160.7|
|   549|    648.2|     99.2|
|   726|    654.5|     71.5|
|   669|    654.3|     14.7|
|   411|    659.4|    248.4|
|   475|    646.2|    171.2|
|   776|    660.2|    115.8|
|   829|    646.6|    182.4|
+------+---------+---------+
only showing top 10 rows


## 6. Residual Analysis

In [7]:
residuals = reg_predictions.withColumn(
    "residual", F.col("CreditScore") - F.col("prediction")
)

print("Residual statistics:")
residuals.select(
    F.round(F.mean("residual"), 4).alias("mean_residual"),
    F.round(F.stddev("residual"), 2).alias("stddev_residual"),
    F.round(F.min("residual"), 2).alias("min_residual"),
    F.round(F.max("residual"), 2).alias("max_residual")
).show()

print("Mean absolute error by churn status:")
residuals.groupBy("Exited").agg(
    F.round(F.mean(F.abs("residual")), 2).alias("mean_abs_error"),
    F.count("*").alias("count")
).orderBy("Exited").show()

print("Actual vs Predicted CreditScore by Geography:")
residuals.groupBy("Geography").agg(
    F.round(F.mean("CreditScore"), 1).alias("avg_actual"),
    F.round(F.mean("prediction"), 1).alias("avg_predicted"),
    F.round(F.mean(F.abs("residual")), 2).alias("mean_abs_error")
).orderBy("Geography").show()

Residual statistics:


+-------------+---------------+------------+------------+
|mean_residual|stddev_residual|min_residual|max_residual|
+-------------+---------------+------------+------------+
|      -2.7662|          97.14|      -260.5|      212.84|
+-------------+---------------+------------+------------+

Mean absolute error by churn status:


+------+--------------+-----+
|Exited|mean_abs_error|count|
+------+--------------+-----+
|     0|         77.33| 1526|
|     1|         83.44|  395|
+------+--------------+-----+

Actual vs Predicted CreditScore by Geography:


+---------+----------+-------------+--------------+
|Geography|avg_actual|avg_predicted|mean_abs_error|
+---------+----------+-------------+--------------+
|   France|     647.0|        650.6|         79.69|
|  Germany|     646.4|        652.6|         78.77|
|    Spain|     653.3|        650.7|         75.97|
+---------+----------+-------------+--------------+



## 7. Export Regression Results for Tableau

In [8]:
reg_export = reg_predictions.select(
    "CreditScore", "prediction",
    "Age", "Balance", "EstimatedSalary", "NumOfProducts",
    "Geography", "Gender", "Exited", "IsActiveMember"
).toPandas()

reg_export.rename(columns={"prediction": "PredictedCreditScore"}, inplace=True)
reg_export["residual"] = reg_export["CreditScore"] - reg_export["PredictedCreditScore"]
reg_export.to_csv("../exports/regression_results.csv", index=False)
print(f"Saved: ../exports/regression_results.csv ({len(reg_export)} rows)")

# Export coefficient table for Tableau
coef_df.to_csv("../exports/regression_coefficients.csv", index=False)
print("Saved: ../exports/regression_coefficients.csv")

Saved: ../exports/regression_results.csv (1921 rows)
Saved: ../exports/regression_coefficients.csv
